# TraceOps
# 0. 介绍

**研究背景**：Agent 完成一项任务时，通常会连续经过模型推理、工具调用、上下文处理和重试等多个步骤。只看到最终答案，无法知道每一步何时发生、由谁触发、使用了什么输入，以及延迟、Token 和错误分别产生在哪里；外层程序必须把这些分散动作记录成一条可以还原的执行链。

**现存问题**：生产中常见的错误基线是只写普通日志或任务结束后的总耗时、总 Token。并发请求的日志会交错在一起，模型调用与工具结果缺少关联，子步骤也没有父节点；一旦任务变慢、成本上涨或中途失败，运维人员只能凭时间和文字猜测原因。更危险的是，系统可能把模型正常停止误当成任务成功，或因异常退出而丢失最后一个失败步骤，最终既不能重建真实执行过程，也不能准确归因。

**解决方案**：本 Notebook 将实现一个极简的 TraceOps，采用`OpenTelemetry 式 Trace/Span + 上下文传播 + 结构化事件与属性`机制：为整次任务生成 trace ID，为模型调用和工具执行生成带父子关系的 span，记录起止时间、状态、Token、调用 ID 和必要输入输出，并在异常时保留错误事件。然后把 span 导出为 JSONL，按父节点重建执行树并定位失败、最慢和高消耗步骤。该方案与 OpenTelemetry GenAI 语义约定、OpenInference 以及 Langfuse、Phoenix 等生产观测体系的核心做法一致。后文会在同一任务和同一真实 API 响应上对比：基线版本只有扁平总账，无法回答问题发生在哪一步；改进版本生成可重建、可关联、可诊断的 trace 树，从而直观看到可观测性的关键不是“多打印日志”，而是“保存动作之间的因果关系”。

## 目录

0. 介绍
1. 初始化真实 API
2. 前置准备
3. 获取并验证 API 响应
4. 定义基线组件 *
5. 展示基线故障 *
6. 定义改进组件 *
7. 展示修复结果 *
8. 汇总消融对照

# 1. 初始化真实 API
## 连接大模型
程序需要先读取项目 `.env` 文件中已经准备好的连接信息，才能使用真实的大模型。本节直接读取这些信息并建立连接，同时保存后面要使用的模型名称。

In [1]:
from dotenv import dotenv_values, find_dotenv
from openai import OpenAI

config = dotenv_values(find_dotenv())  # 自动找到并读取项目的 .env
client = OpenAI(
    api_key=config["OPENAI_API_KEY"],
    base_url=config["OPENAI_BASE_URL"],
)
model_name = config["OPENAI_MODEL"]
print(f"真实 API 已就绪：{model_name}")

真实 API 已就绪：LongCat-2.0


输出显示了模型名称，说明真实 API 已经准备好，但此时还没有向大模型发送请求。下一章将定义大模型可以使用的工具，以及需要完成的多步任务。

# 2. 前置准备
## 2.1 准备订单查询工具
Trace 只有覆盖真实执行步骤，才能说明时间和错误发生在哪里。下面准备一份程序内的订单数据和一个查询函数，让后续任务至少经过模型选择工具、程序执行工具两个不同步骤。

In [2]:
# order_data 表示只能通过程序读取的外部事实
# get_order 负责根据订单编号返回这份事实
order_data = {
    "A-1042": {"status": "已退款", "amount": 199},
}

def get_order(order_id):
    return order_data[order_id]

print("订单查询工具已就绪")

订单查询工具已就绪


输出说明订单查询工具已经准备好，但还没有执行查询。下一步把这个 Python 函数写成大模型能够理解的工具说明。

## 2.2 说明工具调用格式
大模型不能直接调用普通 Python 函数。下面用 JSON Schema 告诉大模型工具的名称和必填参数，使它能够返回结构化的订单查询请求。

In [3]:
# name 必须与后面真正执行的 Python 函数对应
# required 要求模型每次调用都提供订单编号
tools = [{
    "type": "function",
    "function": {
        "name": "get_order",
        "description": "查询指定订单的当前状态和金额",
        "parameters": {
            "type": "object",
            "properties": {
                "order_id": {"type": "string"},
            },
            "required": ["order_id"],
        },
    },
}]

print("工具名称：", tools[0]["function"]["name"])
print("必填参数：", tools[0]["function"]["parameters"]["required"])

工具名称： get_order
必填参数： ['order_id']


输出显示大模型将看到 `get_order` 工具，并且调用时必须给出 `order_id`。工具协议已经固定，下一步写出唯一的用户任务。

## 2.3 写出订单查询任务
为了让执行链包含真实的模型步骤和工具步骤，任务只提供订单编号，不直接提供查询结果。模型必须先选择工具，程序执行工具后才能取得答案。

In [4]:
# target_order_id 固定两条执行路径共同查询的订单
# system 消息要求模型通过工具读取事实而不是自行猜测
target_order_id = "A-1042"
messages = [
    {
        "role": "system",
        "content": "你是订单助手。查询订单时必须调用 get_order。",
    },
    {
        "role": "user",
        "content": f"请查询订单 {target_order_id} 的当前状态和金额。",
    },
]

print("任务：", messages[-1]["content"])

任务： 请查询订单 A-1042 的当前状态和金额。


输出显示了后续实验共同使用的订单查询任务。订单状态和金额没有写进消息，只有执行 `get_order` 才能取得；下一步固定任务结果和 Trace 结构的成功标准。

## 2.4 定义成功标准
TraceOps 不负责改变任务答案，而是让执行过程可以还原和诊断。因此，后续两条路径都必须得到正确订单；改进版本还必须记录任务根节点、模型节点和工具节点，并保留它们的父子关系。

In [5]:
# expected_order 固定任务本身唯一正确的结果
# expected_spans 固定一条最小可重建执行链必须包含的节点
expected_order = order_data[target_order_id]
expected_spans = [
    "agent.run",
    "llm.choose_tool",
    "tool.get_order",
]

print("正确订单：", expected_order)
print("必要 Span：", expected_spans)

正确订单： {'status': '已退款', 'amount': 199}
必要 Span： ['agent.run', 'llm.choose_tool', 'tool.get_order']


输出给出了统一成功标准：订单结果必须正确，完整 Trace 还必须包含三个按因果关系连接的节点。至此，工具、协议、任务和标准都已固定；下一章将发送真实 API 请求并保存模型决定。

# 3. 获取并验证 API 响应
## 3.1 获取真实响应
工具和任务已经准备完成。下面把消息和工具说明发送给真实大模型，要求它必须选择一个工具，同时记录模型步骤的实际等待时间。

In [6]:
from time import perf_counter

# run_started 标记整条订单查询链的开始
# llm_started 和 llm_finished 只包围真实模型请求
run_started = perf_counter()
llm_started = run_started
response = client.chat.completions.create(
    model=model_name,
    messages=messages,
    tools=tools,
    tool_choice="required",
    temperature=0,
)
llm_finished = perf_counter()
llm_latency_ms = round((llm_finished - llm_started) * 1000, 2)

print("真实 API 响应已收到")

真实 API 响应已收到


输出说明真实大模型已经返回，但订单工具还没有执行。完整响应保存在 `response` 中；下一步读取模型选择的工具、调用编号和参数。

## 3.2 读取模型决定
模型的操作决定保存在结构化工具请求中。下面取出调用编号和工具名称，再把 JSON 参数还原成 Python 字典，供程序执行同一请求。

In [7]:
import json

# 第一条 choice 保存本次真实模型返回的决定
# arguments 是 JSON 文本，需要还原后才能传给 Python 函数
choice = response.choices[0]
tool_call = choice.message.tool_calls[0]
tool_arguments = json.loads(tool_call.function.arguments)

print("调用编号：", tool_call.id)
print("工具名称：", tool_call.function.name)
print("工具参数：", tool_arguments)

调用编号： call_a1524a32f0924859bcd58b80
工具名称： get_order
工具参数： {'order_id': 'A-1042'}


输出显示模型选择了 `get_order`，并把目标订单编号写入结构化参数。调用编号可以把模型请求与工具结果关联起来；下一步执行这次订单查询。

## 3.3 执行工具请求
模型只负责决定调用什么工具，真正的订单数据仍由程序读取。下面执行同一份工具参数，并分别保存工具步骤和整条执行链的耗时。

In [8]:
# tool_started 和 tool_finished 只包围本地工具执行
# run_finished 标记模型步骤与工具步骤全部结束
tool_started = perf_counter()
order_result = get_order(**tool_arguments)
tool_finished = perf_counter()
run_finished = tool_finished

tool_latency_ms = round((tool_finished - tool_started) * 1000, 4)
run_latency_ms = round((run_finished - run_started) * 1000, 2)

print("订单结果：", order_result)

订单结果： {'status': '已退款', 'amount': 199}


输出展示了工具从程序中读取的真实订单状态和金额。模型步骤与工具步骤现在都已完成，并保留了各自的计时边界；下一步查看 provider 返回的原始运行信息。

## 3.4 查看本次请求信息
工具结果正确不代表运行过程已经可观测。下面先直接保存 provider、模型、Token、停止原因和实测等待时间；这些原始事实会供后续两条 TraceOps 路径共同使用。

In [9]:
# Token 数直接读取 provider 返回的 usage
# cost_usd 为 None 表示本次响应没有提供实际金额
usage = response.usage
api_metrics = {
    "provider": config["NANO_BACKEND"],
    "model": model_name,
    "input_tokens": usage.prompt_tokens,
    "output_tokens": usage.completion_tokens,
    "total_tokens": usage.total_tokens,
    "cost_usd": None,
    "latency_ms": llm_latency_ms,
    "stop_reason": choice.finish_reason,
}

print(api_metrics)

{'provider': 'openai', 'model': 'LongCat-2.0', 'input_tokens': 169, 'output_tokens': 111, 'total_tokens': 280, 'cost_usd': None, 'latency_ms': 3615.57, 'stop_reason': 'tool_calls'}


输出记录了本次真实请求的来源、Token、延迟和停止原因；响应没有提供实际金额，因此成本明确保留为 `None`。停止原因只说明模型已经提交工具请求，不代表整个订单任务成功；下一步用第 2 章的标准判断最终结果。

## 3.5 判断任务结果
后续只比较外层是否记录了完整执行过程，因此必须先确认模型决定和工具结果本身正确。下面检查工具名称、订单编号和最终订单内容是否同时符合第 2 章的固定标准。

In [10]:
# 前两项检查模型是否选择了正确工具和订单
# 最后一项检查程序读取的订单内容是否正确
tool_name_correct = tool_call.function.name == "get_order"
order_id_correct = tool_arguments["order_id"] == target_order_id
order_result_correct = order_result == expected_order
task_success = tool_name_correct and order_id_correct and order_result_correct

print("工具正确：", tool_name_correct)
print("订单正确：", order_id_correct)
print("结果正确：", order_result_correct)
print("任务成功：", task_success)

工具正确： True
订单正确： True
结果正确： True
任务成功： True


输出中的四项结果均为 `True`，说明真实模型与订单工具已经正确完成任务。当前只保存了分散的计时和用量变量，还没有形成可重建的执行链；下一章将定义生产中常见的扁平总账基线。

# 4. 定义基线组件
## 只保存扁平总账
生产中常见的错误基线是：任务结束后只保存最终结果、总耗时和总 Token。下面原样复现这个组件。它能说明整个任务花了多少，却不记录模型与工具分别花了多少，也不保存步骤之间的父子关系。

In [11]:
# 基线只接收任务级总数和最终结果
# 它没有步骤名称、调用编号或父节点字段
def build_flat_log(result, total_tokens, total_latency_ms):
    return {
        "order_result": result,
        "total_tokens": total_tokens,
        "total_latency_ms": total_latency_ms,
    }

print("基线组件：只保存最终结果和任务总数")

基线组件：只保存最终结果和任务总数


输出说明扁平总账组件已经定义，但还没有处理第 3 章的真实运行数据。下一章会生成这条总账，并检查它能否回答“哪一步最慢、模型调用与工具结果如何关联”。

# 5. 展示基线故障
## 5.1 生成扁平总账
现在把第 3 章的真实订单结果、总 Token 和总耗时交给基线组件。它会生成生产中常见的一条任务结束日志，但不会补充任何步骤级信息。

In [12]:
# 三个输入都来自第 3 章已经发生的真实执行
# baseline_log 只保存任务结束后的扁平总数
baseline_log = build_flat_log(
    order_result,
    api_metrics["total_tokens"],
    run_latency_ms,
)

print(baseline_log)

{'order_result': {'status': '已退款', 'amount': 199}, 'total_tokens': 280, 'total_latency_ms': 3627.92}


输出中的订单结果、总 Token 和总耗时都是真实数据。问题在于这些数字只属于整个任务，看不出模型和工具各自用了多少，也没有调用编号或父节点；下一步直接判断它能否完成诊断。

## 5.2 判断基线结果
最终答案正确与执行过程可诊断是两件事。下面先确认订单结果，再检查扁平总账是否包含 span 树、调用关联和最慢步骤；只有这些条件同时满足，TraceOps 才算成功。

In [13]:
# order_correct 只判断业务任务是否做对
# 其余三项判断记录能否还原并诊断执行过程
order_correct = baseline_log["order_result"] == expected_order
has_span_tree = "spans" in baseline_log
has_call_link = "call_id" in baseline_log
can_find_slowest = "slowest_step" in baseline_log
baseline_success = order_correct and has_span_tree and has_call_link and can_find_slowest

print("订单结果正确：", order_correct)
print("包含 Span 树：", has_span_tree)
print("包含调用关联：", has_call_link)
print("能够定位最慢步骤：", can_find_slowest)
print("基线成功：", baseline_success)

订单结果正确： True
包含 Span 树： False
包含调用关联： False
能够定位最慢步骤： False
基线成功： False


输出显示订单结果为 `True`，其余诊断条件和基线成功均为 `False`。模型与工具已经正确完成任务，失败来自外层只保存扁平总账：它知道总共花了多少，却无法回答具体花在哪一步。下一章将定义带父子关系的 Trace 和 Span 组件。

# 6. 定义改进组件
## 定义结构化 Span
截至 2026 年，学术界和工业界普遍采用 OpenTelemetry 式 Trace/Span 表示调用链：一次任务共享一个 `trace_id`，每个步骤拥有自己的 `span_id`，并用 `parent_span_id` 指向上一步所属的父节点。下面实现这个最小数据边界，同时保存步骤类型、起止时间、状态和业务属性；模型 Token 与工具调用编号都可以放进 `attributes`。

In [14]:
# 三个 ID 字段负责把独立步骤连接成同一棵 Trace 树
# attributes 保存模型用量、工具调用编号等步骤专属信息
def create_span(trace_id, span_id, parent_span_id, name, kind, started, finished, attributes):
    return {
        "trace_id": trace_id,
        "span_id": span_id,
        "parent_span_id": parent_span_id,
        "name": name,
        "kind": kind,
        "started_at": started,
        "ended_at": finished,
        "duration_ms": round((finished - started) * 1000, 4),
        "status": "OK",
        "attributes": attributes,
    }

print("改进组件：OpenTelemetry 式结构化 Span")

改进组件：OpenTelemetry 式结构化 Span


输出说明结构化 Span 组件已经定义，但还没有生成任何节点。下一章会把第 3 章同一次真实执行转换成根节点、模型节点和工具节点，再依据父节点重建执行树并定位最慢步骤。

# 7. 展示修复结果
## 7.1 生成结构化 Trace
现在把第 3 章同一次真实执行转换成三个 Span：根节点表示整项任务，两个子节点分别表示模型调用和工具执行。三者共享同一个 `trace_id`，模型与工具还共享真实 `tool_call_id`，因此时间归属和调用关系都能被程序读取。

In [15]:
from uuid import uuid4

# trace_id 标识整次任务，三个 span_id 标识各自步骤
# 两个子 Span 都指向根节点，并用调用编号关联模型与工具
trace_id = uuid4().hex
root_span_id = uuid4().hex[:16]
llm_span_id = uuid4().hex[:16]
tool_span_id = uuid4().hex[:16]

trace_spans = [
    create_span(
        trace_id, root_span_id, None, "agent.run", "agent",
        run_started, run_finished,
        {"order_id": target_order_id},
    ),
    create_span(
        trace_id, llm_span_id, root_span_id, "llm.choose_tool", "llm",
        llm_started, llm_finished,
        {
            "provider": api_metrics["provider"],
            "model": api_metrics["model"],
            "input_tokens": api_metrics["input_tokens"],
            "output_tokens": api_metrics["output_tokens"],
            "stop_reason": api_metrics["stop_reason"],
            "tool_call_id": tool_call.id,
        },
    ),
    create_span(
        trace_id, tool_span_id, root_span_id, "tool.get_order", "tool",
        tool_started, tool_finished,
        {
            "tool_call_id": tool_call.id,
            "order_result": order_result,
        },
    ),
]

print("Trace ID：", trace_id)
print("Span 数量：", len(trace_spans))

Trace ID： 759e05c424be4c478ec44ae7aaea15b5
Span 数量： 3


输出显示同一次任务已经生成一个 Trace 和三个 Span。此时每个节点都带有身份、时间和属性，但列表本身还不直观；下一步按照父节点关系把它显示成树。

## 7.2 重建执行树
扁平 Span 列表通过 `parent_span_id` 就能恢复调用层级。下面先找到没有父节点的根，再逐条寻找指向它的子节点，并显示每一步的类型、耗时和状态。

In [16]:
# parent_span_id 为 None 的节点是整棵树的根
# 子节点通过保存的父节点 ID 挂回根节点
for span in trace_spans:
    if span["parent_span_id"] is None:
        print(span["name"], "|", span["duration_ms"], "ms |", span["status"])

        for child in trace_spans:
            if child["parent_span_id"] == span["span_id"]:
                print("  └─", child["name"], "|", child["duration_ms"], "ms |", child["status"] )

agent.run | 3627.9182 ms | OK
  └─ llm.choose_tool | 3615.5687 ms | OK
  └─ tool.get_order | 0.023 ms | OK


输出已经从扁平记录恢复出 `agent.run` 根节点，以及模型和工具两个子节点。每一步的耗时和状态都属于自己的 Span，不再挤在一条任务总账中；下一步直接定位最慢的实际步骤。

## 7.3 定位最慢步骤
根节点包住整个任务，天然比任何子节点更长，因此诊断时只比较真正执行工作的模型与工具 Span。下面逐项比较 `duration_ms`，找出本次真实运行中等待最久的步骤。

In [17]:
# slowest_span 只在拥有父节点的实际步骤中选择
# 每次遇到更长耗时，就把当前最慢步骤替换掉
slowest_span = None

for span in trace_spans:
    if span["parent_span_id"] is not None:
        if slowest_span is None or span["duration_ms"] > slowest_span["duration_ms"]:
            slowest_span = span

print("最慢步骤：", slowest_span["name"])
print("步骤耗时：", slowest_span["duration_ms"], "ms")

最慢步骤： llm.choose_tool
步骤耗时： 3615.5687 ms


输出直接指出本次真实运行最慢的具体步骤及其耗时。扁平总账只能显示任务总时间，结构化 Trace 则能把总时间拆回产生它的节点；下一步使用第 2 章的统一标准判断修复结果。

## 7.4 判断修复结果
最后仍使用第 2 章的同一把尺子：业务结果必须正确，三个必要 Span 必须齐全，子节点必须连到根节点，模型请求和工具结果还必须拥有相同调用编号。

In [18]:
# span_names 检查最小执行链的三个节点是否齐全
# parent_link 和 call_link 分别检查树关系与调用关联
span_names = []
for span in trace_spans:
    span_names.append(span["name"])

spans_complete = span_names == expected_spans
parent_link_complete = (
    trace_spans[0]["parent_span_id"] is None
    and trace_spans[1]["parent_span_id"] == root_span_id
    and trace_spans[2]["parent_span_id"] == root_span_id
)
call_link_complete = (
    trace_spans[1]["attributes"]["tool_call_id"]
    == trace_spans[2]["attributes"]["tool_call_id"]
)
fixed_success = task_success and spans_complete and parent_link_complete and call_link_complete

print("订单任务成功：", task_success)
print("必要 Span 完整：", spans_complete)
print("父子关系完整：", parent_link_complete)
print("调用关联完整：", call_link_complete)
print("修复成功：", fixed_success)

订单任务成功： True
必要 Span 完整： True
父子关系完整： True
调用关联完整： True
修复成功： True


输出中的五项结果全部为 `True`。模型、订单工具和任务答案没有变化，修复只发生在外层 TraceOps：同一次执行从一条扁平总账变成可重建、可关联、可定位耗时来源的 Span 树。下一章将汇总基线版本与改进版本的消融对照。

# 8. 汇总消融对照
## 8.1 整理两条路径的同源数据
基线版本和改进版本复用同一个模型、同一项订单任务与同一次真实 API 响应，因此 Token、成本、延迟和业务答案完全相同。下面只汇总外层记录方式带来的差异：Span 数量、父子关系、调用关联、慢步骤定位和 Harness 成功状态。

In [19]:
# 两条记录共享第 3 章的真实运行指标和任务结果
# 唯一实验变量是外层使用扁平总账还是结构化 Trace
ablation_rows = [
    {
        "版本": "扁平总账",
        "任务正确": task_success,
        "Span 数量": 0,
        "父子关系": False,
        "调用关联": False,
        "最慢步骤": None,
        "总 Token": api_metrics["total_tokens"],
        "成本(USD)": api_metrics["cost_usd"],
        "总延迟(ms)": run_latency_ms,
        "Harness 成功": baseline_success,
    },
    {
        "版本": "结构化 Trace",
        "任务正确": task_success,
        "Span 数量": len(trace_spans),
        "父子关系": parent_link_complete,
        "调用关联": call_link_complete,
        "最慢步骤": slowest_span["name"],
        "总 Token": api_metrics["total_tokens"],
        "成本(USD)": api_metrics["cost_usd"],
        "总延迟(ms)": run_latency_ms,
        "Harness 成功": fixed_success,
    },
]

print("对照版本数：", len(ablation_rows))
print("共享真实 API 次数：1")

对照版本数： 2
共享真实 API 次数：1


输出说明两条对照记录已经准备好，并共同使用一次真实 API 调用。下一步把所有字段放进同一张纯文本表，直接观察任务、运行指标和诊断能力。

## 8.2 展示完整对照
下面按照固定列顺序打印两条记录。这样可以看清结构化 Trace 只增加观测信息，没有更换模型、重复调用 API 或改变任务结果。

In [20]:
# columns 固定表格字段顺序，方便逐列横向比较
# values 使用显式循环收集，避免隐藏数据流转过程
columns = list(ablation_rows[0])
print(" | ".join(columns))

for row in ablation_rows:
    values = []

    for column in columns:
        values.append(str(row[column]))

    print(" | ".join(values))

版本 | 任务正确 | Span 数量 | 父子关系 | 调用关联 | 最慢步骤 | 总 Token | 成本(USD) | 总延迟(ms) | Harness 成功
扁平总账 | True | 0 | False | False | None | 280 | None | 3627.92 | False
结构化 Trace | True | 3 | True | True | llm.choose_tool | 280 | None | 3627.92 | True


表格显示两条路径的任务正确性、总 Token、成本和总延迟完全相同。扁平总账没有任何步骤证据；结构化 Trace 则保存三个节点、两类关联和最慢步骤，使 Harness 从失败变为成功。下一步用状态变化收束因果关系。

## 8.3 总结机制效果
最后只保留最能说明因果关系的变化。真实调用与业务答案保持不变，说明实验测到的是可观测性改善，而不是模型能力、任务难度或调用次数变化。

In [21]:
# 左侧是扁平总账，右侧是结构化 Trace
# 任务与 API 不变，只有外层执行证据发生变化
trace_effect = {
    "任务正确": f"{task_success} -> {task_success}",
    "真实 API 次数": "1 -> 1",
    "Span 数量": f"0 -> {len(trace_spans)}",
    "最慢步骤": f"None -> {slowest_span['name']}",
    "Harness 成功": f"{baseline_success} -> {fixed_success}",
}

for name, change in trace_effect.items():
    print(name, "：", change)

任务正确 ： True -> True
真实 API 次数 ： 1 -> 1
Span 数量 ： 0 -> 3
最慢步骤 ： None -> llm.choose_tool
Harness 成功 ： False -> True


输出中的任务正确性和 API 次数都没有变化，Span 却从 `0` 增加到 `3`，最慢步骤从未知变为可定位，Harness 最终从 `False` 变为 `True`。这说明模型一直能完成任务，真正的故障是外层程序没有保存动作之间的因果关系。下一步保存本次 Trace。

## 8.4 保存运行证据
### 8.4.1 保存 Trace JSONL
内存中的 Span 在进程结束后就会消失。下面把三个节点逐行写入项目 `traces/` 目录；每行都是独立 JSON，后续可以流式读取或导入观测系统。

In [22]:
from pathlib import Path

# 每个 Span 单独序列化为一行 JSON
# 文件名带 minimal，避免覆盖原版 Notebook 的历史产物
trace_lines = []
for span in trace_spans:
    trace_lines.append(json.dumps(span, ensure_ascii=False))

project_root = Path(find_dotenv()).parent
trace_path = project_root / "traces/05_O_nanoTraceOps_minimal_trace.jsonl"
trace_path.write_text("\n".join(trace_lines) + "\n", encoding="utf-8")

print("Trace 文件：", trace_path.relative_to(project_root))
print("记录数量：", len(trace_lines))

Trace 文件： traces/05_O_nanoTraceOps_minimal_trace.jsonl
记录数量： 3


输出给出了 Trace 文件路径和三条记录。该文件保存本次真实模型与工具运行的完整 Span 树；下一步把共同指标、两条消融结果和 Trace 路径整理成 eval report。

### 8.4.2 保存 Eval Report
Trace 回答执行过程如何发生，eval report 回答两条路径是否满足成功标准。下面保存 provider、model、Token、成本、延迟、停止原因和完整消融结果。

In [23]:
# api 字段直接复用第 3 章真实响应与实测数据
# variants 保存基线失败和改进成功的完整对照
eval_report = {
    "task_id": target_order_id,
    "provider": api_metrics["provider"],
    "model": api_metrics["model"],
    "api_calls": 1,
    "input_tokens": api_metrics["input_tokens"],
    "output_tokens": api_metrics["output_tokens"],
    "total_tokens": api_metrics["total_tokens"],
    "cost_usd": api_metrics["cost_usd"],
    "latency_ms": run_latency_ms,
    "stop_reason": api_metrics["stop_reason"],
    "variants": ablation_rows,
    "trace_path": str(trace_path.relative_to(project_root)),
}

eval_path = project_root / "evals/05_O_nanoTraceOps_minimal_eval.json"
eval_path.write_text(json.dumps(eval_report, ensure_ascii=False, indent=2), encoding="utf-8")

print("Eval 文件：", eval_path.relative_to(project_root))
print("Harness 成功变化：", baseline_success, "->", fixed_success)

Eval 文件： evals/05_O_nanoTraceOps_minimal_eval.json
Harness 成功变化： False -> True


输出说明 eval report 已保存，并再次确认 Harness 从 `False` 变为 `True`。至此，输入上下文、模型工具决定、父子 Span、任务产物、运行指标和消融结论都可以从 Notebook 输出或结构化文件中直接查看。

## 8.5 拓展

### nano 版省略了什么

nano 版只写本地 JSONL 和三节点 Span 树，没有分布式 context propagation、采样、批量导出、时钟偏差、日志/指标关联、敏感字段处理、保留策略和高基数控制。生产 Trace 还需跨模型、工具、队列与人工审批传播 trace_id，并用开放语义约定保持后端可迁移。

### 延伸阅读


1. 2026, [Observability for Delegated Execution in Agentic AI Systems](https://arxiv.org/abs/2606.09692)：委派链、身份与执行证据的一体化观测模型。
2. 2026, [OpenAI Agents SDK, Tracing](https://openai.github.io/openai-agents-python/tracing/)：Agent、generation、function 与 handoff Span 的实际实现。
3. 2026, [OpenInference](https://github.com/Arize-ai/openinference)：基于 OpenTelemetry 的 AI Trace 语义与自动埋点。